# Gás do Povo — DAG para Avaliação de Impacto (v4)

Este notebook constrói o Directed Acyclic Graph (DAG) metodológico para uma **futura avaliação de impacto** do programa Gás do Povo.

**Correções v4 (adequação aos requisitos da atividade IDP):**
- Adicionado nó **C: Recadastramento no CadÚnico** como **colisor explícito** (X → C ← Y).  
  Condicionar em C (ex.: filtrar apenas famílias com cadastro atualizado) **abre** um caminho espúrio X ↔ Y, viesiando qualquer estimativa — portanto C não deve ser controlado.
- DAG agora cobre todos os requisitos: tratamento (T), desfecho (Y), ≥ 3 confundidores (X, U₁, U₂), 1 mediador (M) e **1 colisor (C)**.

**Correções v3 (auditoria metodológica anterior):**
- Adicionado nó E (elegibilidade formal), separando elegibilidade (ITT) de recebimento efetivo (TOT/LATE).
- Adicionada aresta U₂→Y, tornando U₂ verdadeiro confundidor.
- Corrigido rótulo do back-door (T←E←X→Y, não X→T→Y).
- Estratégias DiD corrigidas para ATT; adicionada coluna de suposições; DiD escalonado com Callaway-Sant'Anna.

**Correções v2 (primeira auditoria):**
- Removido nó "Dados adicionais necessários" (não é variável no DGP).
- Adicionados confundidores latentes U₁ e U₂.
- Corrigida ambiguidade do rótulo "renda".

In [ ]:
!pip install matplotlib>=3.8.0 numpy>=1.26.0 --quiet

In [ ]:
# Gera o DAG executando o script (fonte unica de verdade — C2)
# O script gerar_dag_gas_do_povo.py contem toda a logica de desenho.
# Importar aqui evita duplicacao de codigo entre notebook e script.

DAG_PATH = "dag_gas_do_povo.png"

In [ ]:
%run gerar_dag_gas_do_povo.py

from IPython.display import Image
Image(DAG_PATH)

## Interpretacao do DAG v4

### Atendimento aos requisitos da atividade

| Requisito (atividade IDP) | Nó/Elemento | Status |
|---------------------------|-------------|--------|
| Política pública | Gás do Povo (Lei nº 15.348/2026) | ✅ |
| Tratamento | **T**: voucher mensal de gás | ✅ |
| Desfecho | **Y**: SAN, saúde respiratória, combustível limpo | ✅ |
| ≥ 3 confundidores | **X** (renda/CadÚnico), **U₁** (mercado GLP), **U₂** (fricções acesso) | ✅ |
| 1 mediador | **M**: acesso a gás, redução de biomassa, renda liberada | ✅ |
| **1 colisor** | **C**: Recadastramento no CadÚnico (X→C←Y) | ✅ |
| Variáveis a controlar | Z = {renda per cap., região, BPC, chefia feminina} — via back-door em X | ✅ |

---

### Unidade de analise

O DAG representa a logica causal ao nivel da **familia beneficiaria**. Os dados disponiveis nos notebooks 01 e 02 sao agregados no nivel **municipio x mes** — o que implica risco de falacia ecologica: associacoes observadas entre municipios podem nao refletir as relacoes causais individuais. Nesse nivel, X passa a ser composicao municipal (% chefia feminina, % BPC, renda media per capita).

### Nos e seus papeis causais

| Simbolo | No | Papel | Variaveis representadas |
|---------|-----|-------|------------------------|
| **X** | Caracteristicas pre-tratamento | Confundidor observado | Renda per cap. pre-trat., CadUnico, BPC, chefia feminina, regiao |
| **E** | Elegibilidade formal | Variavel de acesso (intermediaria) | Renda <= 1/2 SM inscrita no CadUnico |
| **T** | Gas do Povo (voucher) | Tratamento | Recebimento e uso efetivo do voucher |
| **Y** | Desfecho social | Resultado causal de interesse | SAN, saude respiratoria, combustivel limpo |
| **C** | Recadastramento no CadUnico | **COLISOR** (X→C←Y) | Atualizacao cadastral motivada por caract. pre-existentes (X) e mudancas nos desfechos (Y) |
| **M** | Mecanismos (mediadores) | Mediadores pos-tratamento | Acesso a gas, reducao de biomassa, renda domiciliar liberada |
| **U1** | Mercado local de GLP | Confundidor latente | Preco regional, densidade de revendedores credenciados |
| **U2** | Friccoes de acesso | Confundidor latente | Alfabetizacao digital, smartphone, distancia ao revendedor |

> **Convencao:** bordas solidas = variaveis observadas; bordas tracejadas = latentes; rosa = colisor.

### O que controlar — justificado pelo DAG

| Variavel | Controlar? | Justificativa pelo DAG |
|----------|------------|------------------------|
| **X** (renda, BPC, chefia fem., regiao) | **Sim** | X é confundidor observado; ajustar por Z fecha o back-door T←E←X→Y |
| **C** (recadastramento CadUnico) | **Nao** | C é colisor (X→C←Y); condicionar em C abre caminho espurio X↔Y, induzindo viés de seleção |
| **M** (mecanismos) | **Nao** (para efeito total) | M é mediador pos-tratamento; controlar por M bloqueia o caminho causal T→M→Y e subestima o efeito total. Controlar apenas em análise de mediacao (efeito direto/indireto) |
| **E** (elegibilidade) | **Opcional** | E está na cadeia causal (E→T); controlar por E em RDD fuzzy separa ITT de LATE |
| **U1**, **U2** | Nao diretamente (latentes) | Exigem variáveis instrumentais; nao sao observados |

### Back-door paths de T para Y

| Caminho | Tipo | Bloqueado por |
|---------|------|---------------|
| T ← E ← X → Y | Back-door (X confundidor) | Ajustar por Z = {renda per cap., regiao, BPC, chefia feminina} |
| T ← U1 → Y | Back-door latente | IV para U1 |
| T ← U2 → Y | Back-door latente | IV para U2 |

### Estrategias de identificacao

| Desenho | Suposicao-chave | Estimando |
|---------|-----------------|-----------|
| **DiD** (Callaway-Sant'Anna) | Tendencias paralelas; rollout escalonado | ATT |
| **PSM** | Ignorabilidade condicional + sobreposicao | ATT |
| **RDD fuzzy** | Continuidade no corte de 1/2 SM + sem manipulacao da renda declarada | LATE |
| **IV** | Relevancia (F > 10) + exclusao + exogeneidade | LATE |

> **ITT vs. TOT:** E (elegibilidade) != T (recebimento efetivo). O RDD no corte de renda estima o ITT. Para o efeito sobre os que realmente usaram o voucher (TOT/LATE), e necessario IV com E como instrumento para T.

### Notas adicionais

- **SUTVA/spillovers:** Em escala nacional, a demanda em massa por gas pode elevar precos locais (T → U1). Isso viola SUTVA e pode atenuar o efeito estimado em analises municipais.
- **Quebra estrutural:** Auxilio Gas (Lei 14.284/2021, deposito bimestral) != Gas do Povo (Lei 15.348/2026, voucher mensal). Ao combinar as series 2021–2026, incluir indicador de fase.
- **DAG estatico:** O grafo e uma versao colapsada por periodo. Um painel municipio x ano exige indexar no tempo (Tt, Yt, Yt-1) e considerar efeitos defasados.